# Bank Marketing: Pipeline of Dataset Group Fairness and Classifier Utility & Group Fairness Results

Author: Ilse Harmers \
Last modified: March 17, 2026

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
from snsynth import Synthesizer
import snsynth.transform as tf
import utils
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import recall_score
from sklearn.metrics import confusion_matrix

In [ ]:
# Initializing the synthesizer for preprocessing the datasets with the TableTransformer function. 
synth = Synthesizer.create('pategan', epsilon = 10.0, delta = 1e-05, plot_losses = True)

In [ ]:
# Importing the real training dataset.
bank_train = pd.read_csv("./train-test-datasets/bank-marketing/bank_train.csv")
"""Needed for TabFairGAN."""
#bank_train.loc[bank_train["age"] <= 25, "age"] = 1
#bank_train.loc[bank_train["age"] > 25, "age"] = 0

# One-hot encoding the categorical columns in the training dataset. 
cat_columns = bank_train.select_dtypes(include=['object']).columns.to_list()
train_cat_encoded = utils.one_hot_encode(bank_train[cat_columns], order = [["no", "yes"], ["no", "yes"], ["no", "yes"], ["no", "yes"]])
#print(train_cat_encoded.shape)

# Transforming the continuous columns with the TableTransformer function.
tt = tf.TableTransformer([
    tf.MinMaxTransformer(lower = bank_train["age"].min(), upper = bank_train["age"].max(),
                         negative = False), # age; scaling to range (0, 1)
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(bank_train["balance"].min()) + 1)),
                             upper = np.log(bank_train["balance"].max() + 1), 
                             negative = False) # balance; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(bank_train["duration"].max() + 1), 
                             negative = False) # duration; scaling to range (0, 1)
    ]),
    tf.MinMaxTransformer(lower = bank_train["campaign"].min(), upper = bank_train["campaign"].max(),
                         negative = False), # campaign; scaling to range (0, 1)
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(bank_train["pdays"].min()) + 1)), 
                             upper = np.log(bank_train["pdays"].max() + 1),
                             negative = False) # pdays; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(bank_train["previous"].max() + 1),
                             negative = False) # previous; scaling to range (0, 1)
    ]),
])

tf_cont_cols = np.array(synth._get_train_data(
                    bank_train[["age", "balance", "duration", "campaign", "pdays", "previous"]], transformer = tt, 
                    preprocessor_eps = 0.0, style = 'gan', nullable = False, categorical_columns=None, 
                    ordinal_columns=None, continuous_columns=None,))

num_encoded = pd.DataFrame(tf_cont_cols, columns = ["age", "balance", "duration", "campaign", "pdays", "previous"])

# Concatenating all the columns back together.
train_encoded = pd.concat([num_encoded.reset_index(drop = True), train_cat_encoded.reset_index(drop = True), 
                           bank_train[["day"]].reset_index(drop = True)], axis = 1)

#train_encoded

In [ ]:
# Importing the real test set.
bank_test = pd.read_csv("./train-test-datasets/bank-marketing/bank_test.csv")
"""Needed for TabFairGAN."""
#bank_test.loc[bank_test["age"] <= 25, "age"] = 1
#bank_test.loc[bank_test["age"] > 25, "age"] = 0

# One-hot encoding the categorical columns in the test dataset. 
test_cat_encoded = utils.one_hot_encode(bank_test[cat_columns], order = [["no", "yes"], ["no", "yes"], ["no", "yes"],
                                        ["no", "yes"]])
#print(test_cat_encoded.shape)

# Transforming the continuous columns with the TableTransformer function.
tt = tf.TableTransformer([
    tf.MinMaxTransformer(lower = bank_test["age"].min(), upper = bank_test["age"].max(),
                         negative = False), # age; scaling to range (0, 1)
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(bank_test["balance"].min()) + 1)),
                             upper = np.log(bank_test["balance"].max() + 1), 
                             negative = False) # balance; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(bank_test["duration"].max() + 1), 
                             negative = False) # duration; scaling to range (0, 1)
    ]),
    tf.MinMaxTransformer(lower = bank_test["campaign"].min(), upper = bank_test["campaign"].max(),
                         negative = False), # campaign; scaling to range (0, 1)
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(bank_test["pdays"].min()) + 1)), 
                             upper = np.log(bank_test["pdays"].max() + 1),
                             negative = False) # pdays; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(bank_test["previous"].max() + 1),
                             negative = False) # previous; scaling to range (0, 1)
    ]),
])

tf_cont_cols = np.array(synth._get_train_data(
                    bank_test[["age", "balance", "duration", "campaign", "pdays", "previous"]], transformer = tt, 
                    preprocessor_eps = 0.0, style = 'gan', nullable = False, categorical_columns=None, 
                    ordinal_columns=None, continuous_columns=None,))

num_encoded = pd.DataFrame(tf_cont_cols, columns = ["age", "balance", "duration", "campaign", "pdays", "previous"])

# Concatenating all the columns back together.
test_encoded = pd.concat([num_encoded.reset_index(drop = True), test_cat_encoded.reset_index(drop = True), 
                          bank_test[["day"]].reset_index(drop = True)], axis = 1)

#test_encoded

In [ ]:
# Setting up pipeline variables.
epsi = [1, 2, 5, 8]   # or [0], if the GAN in question is not differentially private, or ["real"], if calling the real dataset
runs = range(1, 6)  
samples = range(1, 4)

# Initializing file path to the synthetic datasets.
#file_path = "./train-test-datasets/bank-marketing/"
#file_path = "./synthetic-datasets_DP-GAN_B=512/bank-marketing/"
#file_path = "./synthetic-datasets_DPCTGAN_B=512/bank-marketing/"
#file_path = "./synthetic-datasets_FairDP-GAN(dem)_B=512/bank-marketing/"
#file_path = "./synthetic-datasets_FairDP-GAN(dis)_B=512/bank-marketing/"
#file_path = "./synthetic-datasets_FairDP-GAN(dem-dis)_B=512/bank-marketing/"
#file_path = "./synthetic-datasets_TabFair/bank-marketing/"

# Initializing our classifiers.
rf = RandomForestClassifier()
mlp = MLPClassifier(max_iter = 500)
lda = LinearDiscriminantAnalysis()

# Initializing dictionary for confusion matrix title.
clfs = {rf: "rf", mlp: "mlp", lda: "lda"}
clf_names = {rf: "Random Forest Classifier", mlp: "Multi-Layer Perceptron Classifier", lda: "LDA Classifier"}

In [ ]:
# This cell contains the pipeline with which the dataset group fairness, classifier group fairness and classifier utility results are computed.
# These results will be stored as csv files in the corresponding folders (see 'filepath'); these folders must exist for this pipeline to work properly.
for e in epsi:
    for r in runs:
        print(f"Run {r}")
        results = {"dem-parity": [], "dis-impact": []}
        for s in samples:

            # Importing synthetic dataset.
            if e == 0:
                sample = pd.read_csv(file_path + f"run{r}/sample{s}.csv")
            elif e == "real":   # or copying the real dataset
                sample = bank_train.copy()
            else:
                sample = pd.read_csv(file_path + f"epsi_{e}/run{r}/sample{s}.csv")

            # Encoding the sensitive and target attributes for the fairness analysis.
            sample_y_encoded = utils.one_hot_encode(sample[["y"]], order = [["no", "yes"]])
            sample_age_encoded = sample["age"].copy()
            """ Do not (!) run these two lines if TabFairGAN synthetic datasets are used. """
            #"""
            sample_age_encoded.loc[sample_age_encoded <= 25] = 1
            sample_age_encoded.loc[sample_age_encoded > 25] = 0   # """
            sample_cols_encoded = pd.concat([sample_age_encoded.reset_index(drop = True), sample_y_encoded.reset_index(drop = True)], axis = 1)

            dem_sample = utils.demographic_parity(df = sample_cols_encoded, s = "age", y = "y")
            results["dem-parity"].append(dem_sample)

            dis_sample = utils.disparate_impact(df = sample_cols_encoded, s = "age", y = "y")
            results["dis-impact"].append(dis_sample)

            # One-hot encoding the categorical columns in the synthetic dataset. 
            sample_cat_encoded = utils.one_hot_encode(sample[cat_columns], order = [["no", "yes"]])

            # Checking for missing columns between the train and synthetic sets. 
            missing_cols = list(set(list(train_cat_encoded.columns)) - set(list(sample_cat_encoded.columns)))
            # If there is a missing column, most likely due to a label from the train set not being present in the synthetic set, 
            # then we reinsert a 'zero' column into the synthetic set at the right place to account for this missing label.
            if missing_cols:
                for c in missing_cols:
                    df_col = pd.DataFrame({c: sample["day"]*0})
                    sample_cat_encoded.insert(0, c, value = df_col)

            # Transforming the continuous columns with the TableTransformer function.
            tt = tf.TableTransformer([
                tf.MinMaxTransformer(lower = sample["age"].min(), upper = sample["age"].max(),
                                     negative = False), # age; scaling to range (0, 1)
                tf.ChainTransformer([
                    tf.LogModulusTransformer(),
                    tf.MinMaxTransformer(lower = -1 * (np.log(abs(sample["balance"].min()) + 1)),
                                         upper = np.log(sample["balance"].max() + 1), 
                                         negative = False) # balance; scaling to range (0, 1)
                ]),
                tf.ChainTransformer([
                    tf.LogModulusTransformer(),
                    tf.MinMaxTransformer(lower = 0, upper = np.log(sample["duration"].max() + 1), 
                                         negative = False) # duration; scaling to range (0, 1)
                ]),
                tf.MinMaxTransformer(lower = sample["campaign"].min(), upper = sample["campaign"].max(),
                                     negative = False), # campaign; scaling to range (0, 1)
                tf.ChainTransformer([
                    tf.LogModulusTransformer(),
                    tf.MinMaxTransformer(lower = -1 * (np.log(abs(sample["pdays"].min()) + 1)), 
                                         upper = np.log(sample["pdays"].max() + 1),
                                         negative = False) # pdays; scaling to range (0, 1)
                ]),
                tf.ChainTransformer([
                    tf.LogModulusTransformer(),
                    tf.MinMaxTransformer(lower = 0, upper = np.log(sample["previous"].max() + 1),
                                 negative = False) # previous; scaling to range (0, 1)
                ]),
            ])

            tf_cont_cols = np.array(synth._get_train_data(
                                sample[["age", "balance", "duration", "campaign", "pdays", "previous"]], transformer = tt, 
                                preprocessor_eps = 0.0, style = 'gan', nullable = False, categorical_columns=None, 
                                ordinal_columns=None, continuous_columns=None,))

            num_encoded = pd.DataFrame(tf_cont_cols, columns = ["age", "balance", "duration", "campaign", "pdays", "previous"])

            # Concatenating all the columns back together.
            sample_encoded = pd.concat([num_encoded.reset_index(drop = True), sample_cat_encoded.reset_index(drop = True), 
                                        sample[["day"]].reset_index(drop = True)], axis = 1)
            sample_encoded = sample_encoded[train_encoded.columns]

            # Saving the encoded datasets (real or synthetic) for future objectives.
            if e == 0:
                sample_encoded.to_csv(file_path + f"run{r}/sample{s}_encoded.csv", index = False)
            elif e == "real":
                sample_encoded = train_encoded.copy()
                sample_encoded.to_csv(file_path + "train_encoded.csv", index = False)
                test_encoded.to_csv(file_path + "test_encoded.csv", index = False)
            else:
                sample_encoded.to_csv(file_path + f"epsi_{e}/run{r}/sample{s}_encoded.csv", index = False)
            
            for c in clfs:
                np.random.seed(42)   # setting random seed for reproducibility
                # Training an ML classifier and generating its predictions.
                model_synth = c.fit(sample_encoded.drop(columns = ["y"]), sample_encoded["y"])
                preds_synth = model_synth.predict(test_encoded.drop(columns = ["y"]))

                # Accuracy score.
                acc = accuracy_score(test_encoded["y"], preds_synth)*100
                if s == 1:
                    results[f"{clfs[c]}-acc"] = [acc]
                else:
                    results[f"{clfs[c]}-acc"].append(acc)
                
                # Precision score.
                prec = precision_score(test_encoded["y"], preds_synth)*100
                if s == 1:
                    results[f"{clfs[c]}-prec"] = [prec]
                else:
                    results[f"{clfs[c]}-prec"].append(prec)
                
                # Recall score.
                recall = recall_score(test_encoded["y"], preds_synth)*100
                if s == 1:
                    results[f"{clfs[c]}-recall"] = [recall]
                else:
                    results[f"{clfs[c]}-recall"].append(recall)

                # AUROC score.
                auroc = roc_auc_score(test_encoded["y"], preds_synth)
                if s == 1:
                    results[f"{clfs[c]}-auroc"] = [auroc]
                else:
                    results[f"{clfs[c]}-auroc"].append(auroc)

                # Confusion matrix.
                conf_mat = confusion_matrix(test_encoded["y"], preds_synth)
                # Plotting the confusion matrix of the ML classifier.
                fig = sns.heatmap(conf_mat, annot = True, cmap = "mako", fmt = "g")
                fig.set_title(f"Confusion Matrix of {clf_names[c]} \n ($\epsilon = {e}$; run{r}/sample{s})")
                fig.set_xlabel("Predicted Label")
                fig.set_ylabel("True Label")
                fig.set_xticks(ticks = [0.5, 1.5], labels = ["no", "yes"])   # setting ticks on x-axis to match target labels
                fig.set_yticks(ticks = [0.5, 1.5], labels = ["no", "yes"])   # setting ticks on y-axis to match target labels
                if e == 0:
                    plt.savefig(file_path + f"run{r}/confs/sample{s}_conf_{clfs[c]}.png", dpi=500, bbox_inches='tight', pad_inches=0.1)
                elif e == "real":
                    plt.savefig(file_path + f"confs/conf_{clfs[c]}.png", dpi=500, bbox_inches='tight', pad_inches=0.1)
                else:
                    plt.savefig(file_path + f"epsi_{e}/run{r}/confs/sample{s}_conf_{clfs[c]}.png", dpi=500, bbox_inches='tight', pad_inches=0.1)
                plt.close()
                
                # Creating a DataFrame with the encoded 'age' column and the classifier's predictions.
                preds_synth = pd.Series(preds_synth, name = "predicted_y")
                test_age_encoded = bank_test["age"].copy()
                """ Do not (!) run these two lines if TabFairGAN synthetic datasets are used. """
                #"""
                test_age_encoded.loc[test_age_encoded <= 25] = 1
                test_age_encoded.loc[test_age_encoded > 25] = 0   # """
                test_cols = pd.concat([test_age_encoded.reset_index(drop = True), test_encoded[["y"]].reset_index(drop = True),
                                      preds_synth.reset_index(drop = True)], axis = 1)

                # Demographic parity.
                try:
                    dem_clf = utils.demographic_parity(df = test_cols, s = "age", y = "predicted_y")
                except ZeroDivisionError:
                    dem_clf = "np.nan"
                if s == 1:
                    results[f"{clfs[c]}-dem"] = [dem_clf]
                else:
                    results[f"{clfs[c]}-dem"].append(dem_clf)

                # Disparate impact.
                try:
                    dis_clf = utils.disparate_impact(df = test_cols, s = "age", y = "predicted_y")
                except ZeroDivisionError:
                    dis_clf = "np.nan"
                if s == 1:
                    results[f"{clfs[c]}-dis"] = [dis_clf]
                else:
                    results[f"{clfs[c]}-dis"].append(dis_clf)

                # Equal opportunity.
                eqopp_clf = utils.equal_opportunity(df = test_cols, s = "age", y_true = "y", y_pred = "predicted_y")
                if s == 1:
                    results[f"{clfs[c]}-eqopp"] = [eqopp_clf]
                else:
                    results[f"{clfs[c]}-eqopp"].append(eqopp_clf)

        # Saving the 'results' dictionary as a csv file in the appropriate folder.
        df_results = pd.DataFrame(results).rename(index = {0: "sample1", 1: "sample2", 2: "sample3"})
        if e == 0:
            df_results.to_csv(file_path + f"run{r}/results_run{r}.csv")
        elif e == "real":
            df_results.to_csv(file_path + "results_real.csv")
        else:
            df_results.to_csv(file_path + f"epsi_{e}/run{r}/results_epsi-{e}_run{r}.csv")